In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load annotations
ann = pd.read_csv('../data/pmemo/annotations/static_annotations.csv')
ann = ann[['musicId', 'Arousal(mean)', 'Valence(mean)']]

# Load features
feat = pd.read_csv('../data/pmemo_features_librosa.csv')

# Merge data
df = pd.merge(feat, ann, on='musicId')

# Map quadrants
def get_quadrant(v, a):
    if v >= 0.5 and a >= 0.5: return 'Happy'
    if v < 0.5 and a >= 0.5: return 'Angry'
    if v < 0.5 and a < 0.5: return 'Sad'
    return 'Calm'

df['quadrant_label'] = df.apply(lambda x: get_quadrant(x['Valence(mean)'], x['Arousal(mean)']), axis=1)

print(f"Merged Shape: {df.shape}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define variables
X = df.drop(['musicId', 'Arousal(mean)', 'Valence(mean)', 'quadrant_label'], axis=1)
y = df[['Valence(mean)', 'Arousal(mean)']].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Cast to DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Preprocessing complete.")

In [ ]:
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Define models
regressors = {
    "XGBoost": MultiOutputRegressor(XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6)),
    "LightGBM": MultiOutputRegressor(LGBMRegressor(n_estimators=200, learning_rate=0.1, verbose=-1)),
    "RandomForest": MultiOutputRegressor(RandomForestRegressor(n_estimators=200, random_state=42))
}

# Training comparison
for name, reg in regressors.items():
    reg.fit(X_train_scaled, y_train)
    y_pred = reg.predict(X_test_scaled)
    mae_v = mean_absolute_error(y_test[:, 0], y_pred[:, 0])
    mae_a = mean_absolute_error(y_test[:, 1], y_pred[:, 1])
    print(f"{name} - V: {mae_v:.4f}, A: {mae_a:.4f}")

# Plot circumplex
best_reg = regressors["LightGBM"]
y_pred_best = best_reg.predict(X_test_scaled)
plt.figure(figsize=(6, 6))
plt.scatter(y_test[:, 0], y_test[:, 1], alpha=0.3, label='Actual')
plt.scatter(y_pred_best[:, 0], y_pred_best[:, 1], alpha=0.5, label='Predicted')
plt.axhline(0.5)
plt.axvline(0.5)
plt.legend()
plt.show()

In [ ]:
# Residual analysis
errors_v = y_pred_best[:, 0] - y_test[:, 0]
errors_a = y_pred_best[:, 1] - y_test[:, 1]
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); sns.histplot(errors_v, kde=True)
plt.subplot(1, 2, 2); sns.histplot(errors_a, kde=True)
plt.show()

# MAE per quadrant
test_quads = [get_quadrant(v, a) for v, a in y_test]
error_df = pd.DataFrame({'quadrant': test_quads, 'err_v': np.abs(errors_v), 'err_a': np.abs(errors_a)})
print(error_df.groupby('quadrant').mean())

In [ ]:
import joblib

# Commented out to prevent accidental overwriting during development. Uncomment to export models.

# Export components
# joblib.dump(best_reg, '../models/mood_regressor.pkl')
# joblib.dump(scaler, '../models/mood_scaler.pkl')

print("Export complete.")